# 02 — ISP pipeline, stage by stage

Renders the checkerboard scene and walks it through every ISP stage in the
order fixed by ADR 004, visualizing each intermediate. Uses the default
(uncalibrated) params, so the output shows the artifacts the Phase 4 agent
will calibrate away: vignetting, AWB color cast, demosaic fringing, lifted
noisy blacks.

Note: the early panels are *linear* radiance, so they look darker than a
display-encoded image; gamma (stage 9) is where it brightens to look normal.
Outputs cleared in version control; run top-to-bottom.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sensorforge.isp import bayer, color, demosaic, lens, noise
from sensorforge.isp.params import ISPParams
from sensorforge.sim.camera import SimCamera
from sensorforge.sim.renderer import SimRenderer

SCENE = Path.cwd().parent / "scenes" / "checkerboard.xml"
rng = np.random.default_rng(0)
p = ISPParams()

In [ ]:
cam = SimCamera.from_scene(SCENE)
with SimRenderer(cam) as r:
    linear = r.render()

o, s, c = p.optics, p.sensor, p.color
stages = {"1. linear render": linear}

x = lens.apply_vignetting(linear, o.vignette_strength)
stages["2. vignetting"] = x
x = lens.apply_distortion(
    x, o.radial_k1, o.radial_k2, o.radial_k3, o.tangential_p1, o.tangential_p2
)
stages["3. distortion"] = x

raw = bayer.mosaic_rggb(x)
stages["4. bayer raw"] = raw

e = noise.integrate(raw, s.full_well_e, s.exposure_ms)
e = noise.add_dark_current(e, s.dark_current_e_per_s, s.exposure_ms)
e = noise.apply_shot_noise(e, rng)
e = noise.saturate(e, s.full_well_e)
e = noise.apply_read_noise(e, s.read_noise_e, rng)
stages["5. + noise (raw)"] = e / s.full_well_e

dn = color.to_digital(e, s.full_well_e, s.black_level)
dn = color.apply_awb_raw(dn, c.awb_gain_r, c.awb_gain_g, c.awb_gain_b)
stages["6. black level + AWB"] = dn

rgb = demosaic.demosaic_bilinear(dn)
stages["7. demosaic"] = rgb
rgb = color.apply_ccm(rgb, c.ccm)
stages["8. CCM"] = rgb
rgb = color.apply_gamma(rgb, c.gamma)
stages["9. gamma"] = rgb
stages["10. 8-bit output"] = color.quantize_8bit(rgb)

print("linear:", linear.shape, "-> output:", stages["10. 8-bit output"].shape)

## Every stage

Single-channel stages (raw) are shown grayscale; RGB stages in color. All
panels are clipped to [0, 1] for display.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, (title, img) in zip(axes.ravel(), stages.items(), strict=True):
    disp = img.astype(float)
    if disp.max() > 1.0:  # the uint8 output
        disp = disp / 255.0
    ax.imshow(np.clip(disp, 0, 1), cmap="gray" if img.ndim == 2 else None)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()

## Artifact close-up

A crop over the checker edges makes the bilinear false-color fringing and the
sensor noise grain visible.

In [ ]:
crop = (slice(180, 260), slice(270, 370))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(stages["10. 8-bit output"][crop])
ax[0].set_title("final output (checker crop)")
ax[1].imshow(np.clip(stages["5. + noise (raw)"][crop], 0, 1), cmap="gray")
ax[1].set_title("raw + noise (same crop)")
for a in ax:
    a.axis("off")
plt.tight_layout()